# EDA — Preços quebrados no catálogo (P3)

**Objetivo:** entender se a Silver de produtos e categorias serve para prever o preço esperado e achar preços fora do padrão.

**Perguntas do EDA**
1. Quantos produtos, categorias e marcas existem?
2. Existem `preco_custo` e uma coluna de tamanho na Silver?
3. Como é a distribuição de `preco_lista` (zeros, negativos, extremos)?
4. O preço varia por subcategoria, marca e tamanho de forma previsível?
5. Quantos produtos as regras de preço já marcam (rótulo fraco)?
6. Existe margem negativa (`preco_lista` < `preco_custo`)?

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

load_dotenv(".env")

STORAGE_ACCOUNT = "internshipdatalake"
lakehouse_base = f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net"

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

def ler_delta_spark(caminho):
    return spark.read.format("delta").options(**adls_options).load(caminho)

silver_produtos_path = f"{lakehouse_base}/silver/ecommerce_produtos"
silver_categorias_path = f"{lakehouse_base}/silver/ecommerce_categorias"

df_prod = ler_delta_spark(silver_produtos_path)
df_cat = ler_delta_spark(silver_categorias_path)

print("Produtos:  ", df_prod.count(), "linhas |", len(df_prod.columns), "colunas")
print("Categorias:", df_cat.count(), "linhas |", len(df_cat.columns), "colunas")

print("\n=== Schema: produtos ===")
df_prod.printSchema()

print("=== Schema: categorias ===")
df_cat.printSchema()

In [0]:
print("=== Amostra de produtos ===")
(df_prod
    .select("sku", "nome_produto", "descricao", "preco_lista", "unidade_medida", "nome_marca", "id_categoria", "id_categoria_pai")
    .show(12, truncate=45))

print("=== Unidades de medida ===")
df_prod.groupBy("unidade_medida").count().orderBy(F.desc("count")).show(20, truncate=False)

print("Marcas distintas:", df_prod.select("nome_marca").distinct().count())
print("Categorias distintas em produtos:", df_prod.select("id_categoria").distinct().count())
print("Pais (categoria raiz) distintos em produtos:", df_prod.select("id_categoria_pai").distinct().count())

print("\n=== Amostra de categorias ===")
(df_cat
    .select("id_categoria", "nome_categoria", "id_categoria_pai", "nome_categoria_pai", "tipo_categoria")
    .show(12, truncate=40))

print("=== tipo_categoria ===")
df_cat.groupBy("tipo_categoria").count().show(truncate=False)

In [0]:
total = df_prod.count()
print("Linhas:", total)
print("SKUs distintos:", df_prod.select("sku").distinct().count())
print("Micro-lotes (arquivos) distintos:", df_prod.select("bronze_source_file").distinct().count())

print("\n=== Nulos por coluna ===")
colunas_dado = ["id_categoria", "sku", "nome_produto", "descricao", "preco_lista",
                "unidade_medida", "nome_marca", "is_ativo", "id_categoria_pai"]
nulos = df_prod.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_dado]).collect()[0]
for c in colunas_dado:
    print(f"{c}: {nulos[c]} nulos")

print("\n=== Linhas que FALHAM em cada regra ===")
colunas_regra = sorted([c for c in df_prod.columns if c.startswith("regra_")])
falhas = df_prod.select(
    [F.sum(F.when(F.col(c) == False, 1).otherwise(0)).alias(c) for c in colunas_regra]
).collect()[0]
for c in colunas_regra:
    print(f"{c}: {falhas[c]} falhas ({falhas[c] / total * 100:.2f}%)")

In [0]:
# 5.1 Quantas linhas tem cada micro-lote?
print("=== Linhas por micro-lote ===")
por_lote = df_prod.groupBy("bronze_source_file").count()
por_lote.agg(
    F.min("count").alias("minimo"),
    F.max("count").alias("maximo"),
    F.round(F.avg("count"), 1).alias("media"),
).show()

# 5.2 O mesmo SKU aparece mais de uma vez DENTRO do mesmo micro-lote?
pares = df_prod.select("bronze_source_file", "sku").distinct().count()
print("Pares (micro-lote, SKU) distintos:", pares, "de", df_prod.count(), "linhas")

# 5.3 Quantas vezes cada SKU aparece, e quantos preços diferentes ele tem?
por_sku = df_prod.groupBy("sku").agg(
    F.count("*").alias("n_linhas"),
    F.countDistinct("preco_lista").alias("n_precos"),
    F.countDistinct("id_categoria").alias("n_categorias"),
    F.countDistinct("nome_marca").alias("n_marcas"),
    F.min("preco_lista").alias("preco_min"),
    F.max("preco_lista").alias("preco_max"),
)

print("=== Resumo por SKU ===")
por_sku.agg(
    F.min("n_linhas").alias("linhas_min"),
    F.max("n_linhas").alias("linhas_max"),
    F.round(F.avg("n_linhas"), 1).alias("linhas_media"),
    F.round(F.avg("n_precos"), 1).alias("precos_distintos_media"),
    F.max("n_categorias").alias("max_categorias_por_sku"),
    F.max("n_marcas").alias("max_marcas_por_sku"),
).show()

print("=== 5 SKUs que mais aparecem ===")
por_sku.orderBy(F.desc("n_linhas")).show(5, truncate=False)

In [0]:
# 6.1 Visão geral de preco_lista
print("=== Estatísticas gerais de preco_lista ===")
df_prod.select("preco_lista").summary("count", "min", "25%", "50%", "75%", "max", "mean", "stddev").show()

print("=== Quantos preços zero, negativos e altos ===")
df_prod.select(
    F.sum((F.col("preco_lista") == 0).cast("int")).alias("zero"),
    F.sum((F.col("preco_lista") < 0).cast("int")).alias("negativos"),
    F.sum((F.col("preco_lista") > 100).cast("int")).alias("acima_100"),
    F.sum((F.col("preco_lista") > 500).cast("int")).alias("acima_500"),
    F.sum((F.col("preco_lista") > 1000).cast("int")).alias("acima_1000"),
).show()

# 6.2 Qual é o limite da regra 07?
print("=== Menor e maior preço por resultado da regra 07 ===")
df_prod.groupBy("regra_07_preco_dentro_limite").agg(
    F.count("*").alias("linhas"),
    F.min("preco_lista").alias("menor_preco"),
    F.max("preco_lista").alias("maior_preco"),
).show()

# 6.3 Razão entre o preço e a mediana do próprio SKU (só preços positivos)
positivos = df_prod.filter(F.col("preco_lista") > 0)
mediana_sku = positivos.groupBy("sku").agg(
    F.percentile_approx(F.col("preco_lista").cast("double"), 0.5).alias("mediana_sku")
)
df_razao = (
    positivos.join(mediana_sku, "sku")
    .withColumn("razao", F.col("preco_lista").cast("double") / F.col("mediana_sku"))
)

print("=== Distribuição da razão preço / mediana do SKU ===")
df_razao.select("razao").summary("min", "1%", "5%", "25%", "50%", "75%", "95%", "99%", "max").show()

print("=== Contagem por faixa de razão ===")
df_razao.select(
    F.sum((F.col("razao") < 0.5).cast("int")).alias("abaixo_0_5x"),
    F.sum(((F.col("razao") >= 0.5) & (F.col("razao") <= 2)).cast("int")).alias("entre_0_5x_e_2x"),
    F.sum(((F.col("razao") > 2) & (F.col("razao") <= 10)).cast("int")).alias("entre_2x_e_10x"),
    F.sum((F.col("razao") > 10).cast("int")).alias("acima_10x"),
).show()

In [0]:
from pyspark.sql import Window

# 7.1 Como são os nomes dos arquivos (micro-lotes)?
arquivos = df_prod.select("bronze_source_file").distinct()
print("=== Primeiros 5 nomes (ordem alfabética) ===")
arquivos.orderBy("bronze_source_file").show(5, truncate=False)
print("=== Últimos 5 nomes ===")
arquivos.orderBy(F.desc("bronze_source_file")).show(5, truncate=False)

# 7.2 Razão média por micro-lote (preços positivos até 10x a mediana do SKU)
normais = df_razao.filter(F.col("razao") <= 10)
por_lote_razao = normais.groupBy("bronze_source_file").agg(
    F.avg("razao").alias("media_razao"),
    F.count("*").alias("linhas"),
)

print("=== Variação da razão média entre os micro-lotes ===")
por_lote_razao.agg(
    F.round(F.min("media_razao"), 3).alias("menor"),
    F.round(F.max("media_razao"), 3).alias("maior"),
    F.round(F.stddev("media_razao"), 3).alias("desvio"),
).show()

# 7.3 Juntando em 12 blocos de ~30 lotes, na ordem do nome do arquivo
ordem = Window.orderBy("bronze_source_file")
blocos = por_lote_razao.withColumn("bloco", F.ntile(12).over(ordem))
print("=== Razão média por bloco de tempo ===")
blocos.groupBy("bloco").agg(
    F.count("*").alias("lotes"),
    F.round(F.avg("media_razao"), 3).alias("razao_media"),
    F.min("bronze_source_file").alias("primeiro_arquivo"),
).orderBy("bloco").show(12, truncate=False)

# Conclusões da EDA — P3 Preços quebrados no catálogo

## 1. O que são os dados
- Silver `ecommerce_produtos`: **292.000 linhas = 365 micro-lotes × 800 linhas**, chegados a cada ~2 min entre 01/10 e 15/10/2026 (fluxo "real-time").
- **3.805 SKUs distintos**, cada um aparece 42–123 vezes e nunca duas vezes no mesmo lote. **Grão: (micro-lote, SKU) = uma observação de preço.**
- Sem nulos. Categoria e marca são fixas por SKU. 12 categorias raiz, 123 subcategorias, 239 marcas.
- `preco_custo` **não existe** nos dados ingeridos → a margem fica fora da entrega (decisão do Geovany, 09/11/2026).
- `descricao` é texto genérico repetido → descartada.
- O tamanho está escrito no `nome_produto` (ex.: "Água Mineral Natural 1.5L - Bonafont") e foi extraído de lá. O código do SKU é ambíguo (`15l` = 1,5 L, `10kg` = 10 kg) e 4 SKUs vêm truncados.

## 2. Como é o preço
- Mediana R$ 19,23; média R$ 35,68 (puxada pelos erros); mínimo −R$ 299,99; máximo R$ 14.999,50.
- **Erros visíveis:** 2.258 preços zero + 2.113 negativos (= 4.371, regra 02, 1,50%); **2.180 preços acima de 10× a mediana do próprio SKU**; 1.129 entre 2× e 10× (zona cinzenta).
- A regra 07 (limite ≈ R$ 5.000) pega só 72 linhas: **deixa passar os preços gigantes abaixo do limite**. O rótulo fraco (regras 02 e 07) é incompleto.
- A regra 09 mede vendas, não preço: **não usar como rótulo** do P3.
- **Preço normal = nível do SKU × ruído independente.** 90% dos preços ficam entre 0,37× e 1,69× da mediana do SKU. Correlação com a aparição anterior = 0,018. Sem tendência no tempo (razão média por lote entre 0,96 e 1,06). O ruído varia entre SKUs (desvio de 0,19 a 0,80).

## 3. O que explica o nível de preço de um SKU
- **Subcategoria:** maior ganho (MAE R$ 14,81 → 10,19).
- **Tamanho:** ganho adicional (10,19 → 9,23). Pesa muito em 20 de 49 subcategorias analisadas (Lenços Umedecidos, Rações, Fraldas, Areia Sanitária, Água Sanitária, Sacos de Lixo, Água Mineral), com correlação 0,84–0,94, e quase nada em outras (Molhos, Torradas, Balas). O efeito depende da subcategoria.
- **Categoria raiz:** só Mundo Bebê e Pets têm nível claramente mais alto.
- **Marca, Genérico/Marca Própria:** sem efeito mensurável (±2%, menos que o acaso).
- **Unidade (kg/L/un) e tempo:** irrelevantes.
- **Piso de erro:** mesmo conhecendo o SKU, MAE = R$ 9,04 (34%). Subcategoria + tamanho chega a R$ 9,23 (34,7%).

## 4. Decisões para a Engenharia de Features e o modelo
1. **Alvo:** `preco_lista`. Treinar com preços > 0 e sem os gigantes óbvios (ou usar objetivo robusto, tipo L1/Huber).
2. **Features:** subcategoria, categoria raiz, `tamanho`, `log_tamanho`, unidade, tamanho relativo dentro da subcategoria, variante do produto (texto do nome), marca (testar com e sem), flag Genérico/Marca Própria.
3. **Fora:** data/hora, preço anterior, `descricao`, regra 09.
4. **Alerta:** resíduo robusto (MAD) por grupo. O limiar tem que ficar bem acima do ruído de ~±35%.
5. **Avaliação:** comparar com a escadinha de baselines (global, subcategoria, subcategoria + tamanho). Medir Precision/Recall contra o rótulo fraco **e** contra um rótulo de apoio (preço ≤ 0 ou > 10× a mediana do SKU), usado só para avaliar.
6. **Validação:** como as features quase identificam o SKU, separar treino e teste **por SKU** (ou por tempo), e não por linha sorteada.
7. **Expectativa honesta:** o MAE do LightGBM deve ficar perto de R$ 9,2. O valor do modelo está no alerta, e não em prever cada preço.

## 5. Limitações e decisões do Geovany (09/11/2026)
- Sem `preco_custo`, não há cruzamento com margem: a margem fica fora da entrega.
- Não há metas numéricas de sucesso para o P3: reportamos as métricas e comparamos com os baselines.
- O rótulo de apoio (preço ≤ 0 ou > 10× a mediana do SKU) foi aprovado e é usado só na avaliação.

In [0]:
from pyspark.sql import Window

# 8.1 Correlação entre a razão atual e a da aparição anterior do mesmo SKU
por_sku_tempo = Window.partitionBy("sku").orderBy("bronze_source_file")
df_lag = (
    normais
    .withColumn("razao_anterior", F.lag("razao").over(por_sku_tempo))
    .filter(F.col("razao_anterior").isNotNull())
)

print("=== Correlação com a aparição anterior do mesmo SKU ===")
df_lag.select(F.round(F.corr("razao", "razao_anterior"), 4).alias("correlacao")).show()

# 8.2 Tamanho do ruído dentro de cada SKU (desvio padrão da razão)
ruido_sku = normais.groupBy("sku").agg(F.stddev("razao").alias("desvio_razao"))

print("=== Desvio padrão da razão dentro de cada SKU (resumo entre os SKUs) ===")
ruido_sku.agg(
    F.round(F.min("desvio_razao"), 3).alias("menor"),
    F.round(F.avg("desvio_razao"), 3).alias("medio"),
    F.round(F.max("desvio_razao"), 3).alias("maior"),
).show()

In [0]:
# Uma linha por SKU: atributos fixos + nível de preço (mediana)
atributos = (
    df_prod
    .select("sku", "id_categoria", "id_categoria_pai", "unidade_medida", "nome_marca", "nome_produto")
    .dropDuplicates(["sku"])
)
df_sku = atributos.join(mediana_sku, "sku")
print("SKUs com nível de preço calculado:", df_sku.count())

# Nomes das categorias raiz (a tabela repete cada uma 365 vezes, então tiramos duplicatas)
raizes = (
    df_cat.filter(F.col("tipo_categoria") == "Categoria")
    .select(
        F.col("id_categoria").cast("double").alias("id_categoria_pai"),
        F.col("nome_categoria").alias("categoria_raiz"),
    )
    .dropDuplicates(["id_categoria_pai"])
)

# 9.1 Nível de preço por categoria raiz
print("=== Nível de preço dos SKUs por categoria raiz ===")
(
    df_sku.join(raizes, "id_categoria_pai", "left")
    .groupBy("categoria_raiz")
    .agg(
        F.count("*").alias("skus"),
        F.round(F.percentile_approx("mediana_sku", 0.5), 2).alias("mediana_dos_skus"),
        F.round(F.min("mediana_sku"), 2).alias("menor"),
        F.round(F.max("mediana_sku"), 2).alias("maior"),
    )
    .orderBy(F.desc("mediana_dos_skus"))
    .show(15, truncate=False)
)

# 9.2 Nível de preço por unidade de medida
print("=== Nível de preço dos SKUs por unidade ===")
(
    df_sku.groupBy("unidade_medida")
    .agg(
        F.count("*").alias("skus"),
        F.round(F.percentile_approx("mediana_sku", 0.5), 2).alias("mediana_dos_skus"),
        F.round(F.min("mediana_sku"), 2).alias("menor"),
        F.round(F.max("mediana_sku"), 2).alias("maior"),
    )
    .show()
)

# 9.3 Tamanho escondido no SKU (terceiro pedaço entre os hifens)
df_sku = df_sku.withColumn("token_tamanho", F.split("sku", "-").getItem(2))
print("Formatos de tamanho distintos:", df_sku.select("token_tamanho").distinct().count())
print("=== 25 formatos mais comuns ===")
df_sku.groupBy("token_tamanho").count().orderBy(F.desc("count")).show(25, truncate=False)

fora_padrao = df_sku.filter(~F.col("token_tamanho").rlike(r"^\d+(un|kg|g|l|ml)$"))
print("SKUs cujo pedaço NÃO parece um tamanho:", fora_padrao.count())
fora_padrao.select("sku").show(10, truncate=False)

In [0]:
# 10.1 Todos os formatos de tamanho que existem
todos = [r["token_tamanho"] for r in df_sku.select("token_tamanho").distinct().orderBy("token_tamanho").collect()]
print("=== Todos os formatos ===")
print(todos)

# 10.2 Dois exemplos reais para cada formato ambíguo
tokens_para_ver = ["05kg", "5kg", "01kg", "1kg", "02kg", "2kg", "005kg", "035l", "15l", "10kg", "1un", "50un"]
janela = Window.partitionBy("token_tamanho").orderBy("sku")
exemplos = (
    df_sku.filter(F.col("token_tamanho").isin(tokens_para_ver))
    .withColumn("ordem", F.row_number().over(janela))
    .filter(F.col("ordem") <= 2)
    .select(
        "token_tamanho",
        "nome_produto",
        "unidade_medida",
        F.round("mediana_sku", 2).alias("nivel_preco"),
    )
    .orderBy("token_tamanho")
)
print("\n=== Exemplos reais por formato ===")
exemplos.show(40, truncate=False)

# 10.3 Os SKUs que parecem cortados
print("=== SKUs fora do padrão, com o nome completo ===")
fora_padrao.select("sku", "nome_produto", "unidade_medida").show(truncate=False)

In [0]:
# 11.1 Extrair tamanho e unidade do nome do produto (ex.: "... 0.35L - Amstel")
padrao = r"(\d+(?:\.\d+)?)(kg|L|un)\s-\s"
numero = F.regexp_extract("nome_produto", padrao, 1)
unidade = F.regexp_extract("nome_produto", padrao, 2)

df_sku = (
    df_sku
    .withColumn("tamanho", F.when(numero != "", numero.cast("double")))
    .withColumn("unidade_no_nome", F.when(unidade != "", unidade))
)

print("=== Conferência da extração ===")
print("SKUs sem tamanho extraído:", df_sku.filter(F.col("tamanho").isNull()).count())
print("SKUs em que a unidade do nome é diferente de unidade_medida:",
      df_sku.filter(F.col("unidade_no_nome") != F.col("unidade_medida")).count())
df_sku.agg(F.min("tamanho").alias("menor_tamanho"), F.max("tamanho").alias("maior_tamanho")).show()
df_sku.select("nome_produto", "tamanho", "unidade_no_nome").orderBy(F.desc("tamanho")).show(5, truncate=False)

# 11.2 Correlação entre log(tamanho) e nível de preço, dentro de cada subcategoria
subcategorias = (
    df_cat.filter(F.col("tipo_categoria") == "Subcategoria")
    .select("id_categoria", F.col("nome_categoria").alias("subcategoria"))
    .dropDuplicates(["id_categoria"])
)

com_tamanho = df_sku.filter(F.col("tamanho") > 0).withColumn("log_tamanho", F.log("tamanho"))
corr_sub = (
    com_tamanho.groupBy("id_categoria", "unidade_medida")
    .agg(
        F.count("*").alias("skus"),
        F.countDistinct("tamanho").alias("tamanhos_distintos"),
        F.corr("log_tamanho", "mediana_sku").alias("corr"),
    )
    .filter((F.col("skus") >= 8) & (F.col("tamanhos_distintos") >= 3))
)

print("=== Resumo das correlações (subcategorias com pelo menos 8 SKUs e 3 tamanhos) ===")
corr_sub.agg(
    F.count("*").alias("grupos"),
    F.round(F.avg("corr"), 3).alias("corr_media"),
    F.round(F.percentile_approx("corr", 0.5), 3).alias("corr_mediana"),
    F.sum((F.col("corr") > 0.5).cast("int")).alias("acima_de_0_5"),
    F.sum((F.col("corr") < -0.5).cast("int")).alias("abaixo_de_menos_0_5"),
).show()

corr_nome = corr_sub.join(subcategorias, "id_categoria", "left")
print("=== 8 subcategorias onde o tamanho MAIS pesa ===")
corr_nome.select("subcategoria", "skus", F.round("corr", 3).alias("corr")).orderBy(F.desc("corr")).show(8, truncate=False)
print("=== 5 subcategorias onde o tamanho MENOS pesa ===")
corr_nome.select("subcategoria", "skus", F.round("corr", 3).alias("corr")).orderBy("corr").show(5, truncate=False)

In [0]:
# 12.1 Quantos SKUs cada marca tem
por_marca = df_sku.groupBy("nome_marca").agg(F.count("*").alias("skus"))
print("Marcas distintas entre os SKUs:", por_marca.count())
por_marca.agg(
    F.min("skus").alias("minimo"),
    F.percentile_approx("skus", 0.5).alias("mediana"),
    F.max("skus").alias("maximo"),
).show()
print("=== 10 marcas com mais SKUs ===")
por_marca.orderBy(F.desc("skus")).show(10, truncate=False)

# 12.2 Nível do SKU em relação aos vizinhos (mesma subcategoria e mesmo tamanho)
celulas = df_sku.groupBy("id_categoria", "tamanho").agg(
    F.count("*").alias("vizinhos"),
    F.percentile_approx("mediana_sku", 0.5).alias("mediana_celula"),
)
df_rel = (
    df_sku.join(celulas, ["id_categoria", "tamanho"])
    .filter(F.col("vizinhos") >= 4)
    .withColumn("rel", F.col("mediana_sku") / F.col("mediana_celula"))
)

print("=== Quanto os SKUs de mesma subcategoria e tamanho diferem entre si ===")
df_rel.agg(
    F.count("*").alias("skus_comparados"),
    F.round(F.avg("rel"), 3).alias("rel_media"),
    F.round(F.stddev("rel"), 3).alias("rel_desvio"),
    F.round(F.min("rel"), 2).alias("menor"),
    F.round(F.max("rel"), 2).alias("maior"),
).show()

# 12.3 Efeito da marca (marcas com pelo menos 10 SKUs comparados)
efeito_marca = (
    df_rel.groupBy("nome_marca")
    .agg(F.count("*").alias("skus"), F.round(F.avg("rel"), 3).alias("rel_media"))
    .filter(F.col("skus") >= 10)
)
print("=== Quanto as marcas diferem entre si (rel_media por marca) ===")
efeito_marca.agg(
    F.count("*").alias("marcas"),
    F.round(F.min("rel_media"), 3).alias("menor"),
    F.round(F.max("rel_media"), 3).alias("maior"),
    F.round(F.stddev("rel_media"), 3).alias("desvio"),
).show()
print("=== 8 marcas mais caras que os vizinhos ===")
efeito_marca.orderBy(F.desc("rel_media")).show(8, truncate=False)
print("=== 8 marcas mais baratas que os vizinhos ===")
efeito_marca.orderBy("rel_media").show(8, truncate=False)

# 12.4 Genérico e Marca Própria
print("=== Genérico e Marca Própria ===")
(
    df_rel.filter(F.col("nome_marca").isin("Genérico", "Marca Própria"))
    .groupBy("nome_marca")
    .agg(F.count("*").alias("skus"), F.round(F.avg("rel"), 3).alias("rel_media"))
    .show()
)

In [0]:
# Base: preços positivos "normais" com subcategoria, tamanho e mediana do SKU
base = (
    normais
    .select("sku", "id_categoria", "preco_lista", "mediana_sku")
    .join(df_sku.select("sku", "tamanho"), "sku")
    .withColumn("preco", F.col("preco_lista").cast("double"))
)

# Quatro previsores, do mais simples ao mais preciso
mediana_global = base.approxQuantile("preco", [0.5], 0.001)[0]

med_sub = base.groupBy("id_categoria").agg(
    F.percentile_approx("preco", 0.5).alias("pred_subcategoria")
)
med_cel = base.groupBy("id_categoria", "tamanho").agg(
    F.percentile_approx("preco", 0.5).alias("pred_subcat_tamanho")
)

b = (
    base
    .join(med_sub, "id_categoria")
    .join(med_cel, ["id_categoria", "tamanho"])
    .withColumn("pred_global", F.lit(mediana_global))
    .withColumn("pred_sku", F.col("mediana_sku"))
)

print("Linhas avaliadas:", b.count())
print(f"Mediana global usada como chute: R$ {mediana_global:.2f}\n")
print(f"{'Previsor':<22}{'MAE (R$)':>10}{'Erro relativo (%)':>20}")

for nome in ["pred_global", "pred_subcategoria", "pred_subcat_tamanho", "pred_sku"]:
    r = b.agg(
        F.avg(F.abs(F.col("preco") - F.col(nome))).alias("mae"),
        F.avg(F.abs(F.col("preco") - F.col(nome)) / F.col(nome)).alias("rel"),
    ).collect()[0]
    print(f"{nome:<22}{r['mae']:>10.2f}{r['rel'] * 100:>20.1f}")